In [8]:
import pandas as pd
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
import csv

In [9]:
cd C:\Users\angel\Desktop\Recommand-System\MOST_committee

C:\Users\angel\Desktop\Recommand-System\MOST_committee


In [3]:
file_path = 'data/research_proj/115計算機學門審查/(勿對外公開資料或流傳)108-115年智慧計算學門大批專題計畫申請案件(含中英文摘要及關鍵字).xlsx'
apply_project_excel_file = pd.ExcelFile(file_path)
apply_project_df = pd.read_excel(apply_project_excel_file, '115') #審查資料

In [ ]:
# 初始化嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-zh-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# 載入向量資料庫
title_vectorstore = Chroma(
    "titles",
    persist_directory="database/vectorstore_bge_merged",
    embedding_function=embeddings
)

keyword_vectorstore = Chroma(
    "keywords",
    persist_directory="database/vectorstore_bge_merged",
    embedding_function=embeddings
)

C:\Users\angel\AppData\Local\Temp\ipykernel_5064\1497608586.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\angel\anaconda3\envs\MOST\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
C:\Users\angel\AppData\Local\Temp\ipykernel_5064\1497608586.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the

In [ ]:
title_col = "計畫中文名稱"  # 替換成你的標題欄位名稱
keyword_col = "中文關鍵字"  # 替換成你的關鍵字欄位名稱
RECOMMAND_AMOUNT = 10 

# 初始化兩個獨立的 DataFrame 來存儲結果
title_similarity_df = pd.DataFrame(columns=["query_text", "compared_text", "recommended_manager", "model_name", "similarity_score"])
keyword_similarity_df = pd.DataFrame(columns=["query_text", "compared_text", "recommended_manager", "model_name", "similarity_score"])

for project in apply_project_df.itertuples():
    title = getattr(project, title_col)
    keywords = getattr(project, keyword_col)
    keywords_str = keywords.replace('，', ',').replace('；', ',').replace(';', ',').replace('、', ',').replace('。', ',')
    keywords_str = keywords_str.replace('\n', ',').replace('\r', ',')
    
    # 標題相似度搜索
    title_documents = title_vectorstore.similarity_search_with_relevance_scores(
                    title,
                    k=RECOMMAND_AMOUNT
                )
    
    # 關鍵字相似度搜索
    keyword_documents = keyword_vectorstore.similarity_search_with_relevance_scores(
                    keywords_str,
                    k=RECOMMAND_AMOUNT
                )      
    
    # 處理標題搜索結果
    for doc, score in title_documents:
        recommended_manager = doc.metadata['manager'] 
        compared_text = doc.page_content
        model_name = "BGE_ZH"
        
        # 創建新行
        new_row = pd.DataFrame([{
            "query_text": title,
            "compared_text": compared_text,
            "recommended_manager": recommended_manager,
            "model_name": model_name,
            "similarity_score": score
        }])
        
        # 將新行添加到標題結果 DataFrame
        title_similarity_df = pd.concat([title_similarity_df, new_row], ignore_index=True)
    
    # 處理關鍵字搜索結果
    for doc, score in keyword_documents:
        recommended_manager = doc.metadata['manager'] 
        compared_text = doc.page_content
        model_name = "BGE_ZH"
        
        # 創建新行
        new_row = pd.DataFrame([{
            "query_text": keywords_str,  # 這裡改為使用關鍵字作為查詢文本
            "compared_text": compared_text,
            "recommended_manager": recommended_manager,
            "model_name": model_name,
            "similarity_score": score
        }])
        
        # 將新行添加到關鍵字結果 DataFrame
        keyword_similarity_df = pd.concat([keyword_similarity_df, new_row], ignore_index=True)

# 檢查結果數量
print(f"標題相似度結果數量: {len(title_similarity_df)}")
print(f"關鍵字相似度結果數量: {len(keyword_similarity_df)}")

# 將結果保存到 Excel 文件的不同工作表
with pd.ExcelWriter("output/results_bge_merged.xlsx") as writer:
    title_similarity_df.to_excel(writer, index=False, sheet_name='title')
    keyword_similarity_df.to_excel(writer, index=False, sheet_name='keyword')

print("結果已保存到 output/results_bge_merged.xlsx")


C:\Users\angel\AppData\Local\Temp\ipykernel_5064\1692134894.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  title_similarity_df = pd.concat([title_similarity_df, new_row], ignore_index=True)
C:\Users\angel\AppData\Local\Temp\ipykernel_5064\1692134894.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  keyword_similarity_df = pd.concat([keyword_similarity_df, new_row], ignore_index=True)


標題相似度結果數量: 6410
關鍵字相似度結果數量: 6410
結果已保存到 output/results_bge_merged.xlsx
